# Description

In this notebook, I will load the Llama3-2 weight and try quantize it

In [1]:
import os
import sys
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import OrderedDict
import re 

import torch
torch.manual_seed(123)

from utils.model_quantize import Llama3Model, generate, text_to_token_ids, token_ids_to_text 
from utils.tokenizer import Llama3Tokenizer, ChatFormat, clean_text
from utils.model import LLAMA32_CONFIG_1B, LLAMA32_CONFIG_3B
from utils.model_quantize import GroupedQueryAttention

# 1. Load model

In [2]:
# ===== Hyper-parameter =====
MODEL_FILE = "/home/tnguyen10/Desktop/llm/model/llama3.2-1B-instruct.pth"
MODEL_CONTEXT_LENGTH = 8192  # Support up to 131_072
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
TOP_K = 1
TOKENIZER_FILE = "/home/tnguyen10/Desktop/llm/model/tokenizer.model"
DEVICE = "cuda"

In [3]:
# 1. Load model
if os.path.exists(MODEL_FILE) == False:
    print(f"[ERROR] Model does not exist !!!")
    sys.exit(0)

if "1B" in MODEL_FILE:
    llama32_config = LLAMA32_CONFIG_1B
elif "3B" in MODEL_FILE:
    llama32_config = LLAMA32_CONFIG_3B
else:
    print(f"[ERROR] Check model file again !!!")
    sys.exit(0)

llama32_config["context_length"] = MODEL_CONTEXT_LENGTH

model = Llama3Model(llama32_config)
checkpoint = torch.load(MODEL_FILE, weights_only=True, map_location=DEVICE)

In [4]:
def remap_for_parameter_only(ckpt, target_dtype=None, device=None):
    """
    Remap state_dict from nn.Linear-style keys (...W_key.weight) with shape [out,in]
    to parameter-only keys (...W_key) with shape [in,out].
    Same for W_value, W_query, out_proj.
    """
    new_sd = OrderedDict()
    # patterns: replace ".W_key.weight" -> ".W_key" etc.
    patterns = [
        (re.compile(r"\.W_key\.weight$"),   ".W_key"),
        (re.compile(r"\.W_value\.weight$"), ".W_value"),
        (re.compile(r"\.W_query\.weight$"), ".W_query"),
        (re.compile(r"\.out_proj\.weight$"), ".out_proj"),
    ]

    for k, v in ckpt.items():
        new_key = k
        for rgx, repl in patterns:
            if rgx.search(k):
                new_key = rgx.sub(repl, k)  # rename to parameter-only key
                v = v                   # transpose [out,in] -> [in,out]
                break

        # cast if requested
        if target_dtype is not None:
            v = v.to(target_dtype)
        if device is not None:
            v = v.to(device)

        new_sd[new_key] = v

    return new_sd

checkpoint = remap_for_parameter_only(
    checkpoint, device="cuda", target_dtype=torch.float16
)

missing, unexpected = model.load_state_dict(checkpoint)
model.to(DEVICE)

print(f"Model {MODEL_FILE} loaded successfully !!!")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params/1e6:.2f} M")

Model /home/tnguyen10/Desktop/llm/model/llama3.2-1B-instruct.pth loaded successfully !!!
Total parameters: 1498.48 M


In [5]:
# 2. Load tokenizer
if not os.path.exists(TOKENIZER_FILE):
    print(f"[ERROR] Does not have TOKENIZER_FILE")
    sys.exit(0)

tokenizer = Llama3Tokenizer(TOKENIZER_FILE)

if "instruct" in MODEL_FILE:
    tokenizer = ChatFormat(tokenizer)
print(f"Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [6]:
input_prompt = "What is the capital of VietNam?"
print(f"Input prompt: {input_prompt}")
print('-' * 80)

# 3. Generate text
start = time.time()

token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_prompt, tokenizer).to(DEVICE),
    max_new_tokens=MAX_NEW_TOKENS,
    context_size=llama32_config["context_length"],
    top_k=TOP_K,
    temperature=TEMPERATURE,
)

print(f"Generation time: {time.time() - start:.2f} sec")

output_text = token_ids_to_text(token_ids, tokenizer)
output_text = clean_text(output_text)
print("\nOutput text:\n", output_text)

Input prompt: What is the capital of VietNam?
--------------------------------------------------------------------------------

 [WARNING]: Reached limit of 512 token without generating an EOS token. 

Generation time: 7.83 sec

Output text:
 The capital of Vietnam is Hanoi.


# 2. Quantize weight 

In [7]:
# Loop over each GroupedQueryAttention and perform quantization
for name, module in model.named_modules():
    if isinstance(module, GroupedQueryAttention):
        print(f"\nQuantizing weights for module: {name} ...")
        module.quantize_weights()
        print(f"Quantization done for module: {name}.")


Quantizing weights for module: trf_blocks.0.att ...
[INFO] Quantized weights to int8. Deleted original weights to save memory.
[INFO] Activation quantization scale set to 0.036133
[INFO] Output activation quantization scale set to 0.005890
Quantization done for module: trf_blocks.0.att.

Quantizing weights for module: trf_blocks.1.att ...
[INFO] Quantized weights to int8. Deleted original weights to save memory.
[INFO] Activation quantization scale set to 0.041748
[INFO] Output activation quantization scale set to 0.006409
Quantization done for module: trf_blocks.1.att.

Quantizing weights for module: trf_blocks.2.att ...
[INFO] Quantized weights to int8. Deleted original weights to save memory.
[INFO] Activation quantization scale set to 0.060547
[INFO] Output activation quantization scale set to 0.008057
Quantization done for module: trf_blocks.2.att.

Quantizing weights for module: trf_blocks.3.att ...
[INFO] Quantized weights to int8. Deleted original weights to save memory.
[INFO

In [10]:
input_prompt = "What is the Ben Tre province in VietNam?"
print(f"Input prompt: {input_prompt}")
print('-' * 80)

# Run inference again after quantization
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_prompt, tokenizer).to(DEVICE),
    max_new_tokens=MAX_NEW_TOKENS,
    context_size=llama32_config["context_length"],
    top_k=TOP_K,
    temperature=TEMPERATURE,
)

output_text = token_ids_to_text(token_ids, tokenizer)
output_text = clean_text(output_text)
print("\nOutput text after quantization:\n", output_text)

Input prompt: What is the Ben Tre province in VietNam?
--------------------------------------------------------------------------------

 [WARNING]: Reached limit of 512 token without generating an EOS token. 


Output text after quantization:
 Ben Tre is a province located in the southern part of Vietnam, near the Mekong Delta. It is one of the 13 provinces of the Mekong Delta region, which is a major agricultural and industrial hub in Vietnam.

Ben Tre Province is situated about 100 kilometers (62 miles) south of Ho Chi Minh City, the capital of Vietnam. The province is known for its rich agricultural land, with a focus on rice, coffee, and other crops. The province is also home to a diverse range of wildlife, including monkeys, birds, and other animals.

Ben Tre is a popular tourist destination, with many attractions and activities to enjoy, such as:

* The Ben Tre Market, which is one of the largest markets in Vietnam
* The Ben Tre Museum, which showcases the province's history and

In [11]:
# Make sure the scale value are scalar 
idx_block = 0
print(f"W_query scale: {model.trf_blocks[idx_block].att.W_query_scale}")
print(f"W_key scale: {model.trf_blocks[idx_block].att.W_key_scale}")
print(f"W_value scale: {model.trf_blocks[idx_block].att.W_value_scale}")
print(f"W_out scale: {model.trf_blocks[idx_block].att.out_proj_scale}")
print(f"x scale: {model.trf_blocks[idx_block].att.x_scale}")
print(f"x_o scale: {model.trf_blocks[idx_block].att.x_o_scale}")

W_query scale: 0.00531005859375
W_key scale: 0.00506591796875
W_value scale: 0.000514984130859375
W_out scale: 0.002471923828125
x scale: 0.0361328125
x_o scale: 0.005889892578125


In [ ]:
# # Save model state_dict after quantization to file and measure size
# quantized_model_file = "quantized_model.pth"
# torch.save(model.state_dict(), quantized_model_file)
# original_size = os.path.getsize(MODEL_FILE) / (1024 * 1024)
# quantized_size = os.path.getsize(quantized_model_file) / (1024 * 1024)
# print(f"\nOriginal model size: {original_size:.2f} MB")
# print(f"Quantized model size: {quantized_size:.2f} MB")